# Hamiltonian evolution with QSVT

Regenerate the cosine and sine phases, combine both responses and compare the projected vector against an exact matrix exponential.

## Follow the actual quantum construction

`workflow.encoding()` builds a native Circuit with `SelectPauliLCU` between
amplitude preparation and its adjoint. `workflow.run()` fits phases, constructs
the cosine/sine response matrices and executes their six-qubit selection
Circuit. Inspect [encoding, selected_response and evolution_circuit](workflow.py)
for the construction. This workflow uses real FlagQuantum execution, but its
QSVT response is composed classically into a dense matrix passed to Circuit
`any`; it is not an integrated high-level qfunc QSVT synthesis path.

Consequently, the represented operation count includes an opaque matrix
operation. It cannot be compared directly with a decomposed sequence of
one- and two-qubit gates or used to claim scalable gate synthesis. The dense
reference also limits this particular implementation to a small block.

Try a different seed through `workflow.run(seed=...)`, preserving the original
Hamiltonian, time and error target. The expected complex state should change
while the full-state and evolution checks still pass. Read postselection
probability separately from the rescaled vector error; zero-block projection
does not imply every flag is clean across the complete state. Phase fitting
and matrix-reference accuracy are separate from quantum-execution accuracy.


Use the local FlagQuantum environment.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq

Define the four-term Hamiltonian, evolution time and tolerance.

In [ ]:
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qsvt.overview import (
    workflow,
)
from examples.synthesis.algorithms.hamiltonian_simulation.hamiltonian_simulation_with_block_encoding.hamiltonian_simulation_qsvt.overview.main import (
    print_report,
    save_figures,
)

print(fq.__name__)
print(workflow.WORDS, workflow.WEIGHTS, workflow.TIME, workflow.EPS)

Construct the checked Pauli LCU block from its four terms.

In [ ]:
block = workflow.encoding()
print(block.target, block.alpha)

Generate one reproducible real input and normalize it.

In [ ]:
amplitudes = workflow.input_state()
print(amplitudes)

Inspect the two extra flags for extracting and combining the QSVT responses.

In [ ]:
print("Data, Hamiltonian block, response and LCU widths:", 2, 2, 1, 1)

Build the Jacobi–Anger polynomials at their original degree and scale.

In [ ]:
print("Series degree and scaling:", workflow.DEGREE, workflow.SCALE)

Apply the full block and the combined cosine and sine responses.

In [ ]:
result = workflow.run()
print_report(result)

Verify the Hamiltonian block projection.

In [ ]:
assert result.block_error < 1e-9

Check the alternating even and odd phase streams.

In [ ]:
assert abs(result.cosine.phases.numel() - result.sine.phases.numel()) <= 1

Read the complete state comparison and the represented circuit width.

In [ ]:
print("Full complex-state error:", result.full_state_error)
print("Circuit width:", result.resources.n_qubits)

Inspect the zero-block projection and rescale by the original response factor.

In [ ]:
print("Projection probability and rescale factor:", result.success_probability, 2 / workflow.SCALE)

Compare against the matrix exponential and display both plots.

In [ ]:
assert result.evolution_error < workflow.EPS
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "qsvt-response-errors.png")))
display(Image(filename=str(figures / "qsvt-evolution-state.png")))